# Superfermion vs Qiskit: Master CPU Benchmark

| Method | Superfermion | Qiskit Aer |
|--------|-------------|------------|
| Statevector | Rust + Rayon + in-place | C++ (Aer) |
| MPS | Rust tensor network | C++ MPS |
| Stabilizer | Rust tableau | C++ stabilizer |

**Methodology:**
- Same circuit, same parameters, same qubit counts
- 15 trials per data point, median reported; all raw per-trial times saved to JSON
- Global warmup at largest size + **per-size warmup** (throwaway call at each n/depth before timed trials to eliminate cold-start artifacts from thread pools, JIT, etc.)
- Correctness verified for **every method** before timing:
  - Statevector: cross-framework fidelity (direct comparison — both use LSB qubit ordering)
  - Stabilizer: TVD of samples against exact statevector distribution
  - MPS: GHZ distribution check (must produce ~50/50 on |0...0> and |1...1>)
- GC disabled during timing to reduce measurement noise

In [1]:
# -- Bootstrap: pin the fixed LOCAL build (added 2026-09-24) ----------------
# Numbers in this notebook must describe the fixed local build, not any
# globally installed wheel, so the repo is pinned ahead of site-packages.
import sys as _sys
import hashlib as _hashlib
from pathlib import Path as _Path
REPO = _Path(r"C:\Users\ASUS\OneDrive\Desktop\sfdocs\superfermion")
assert (REPO / 'superfermion' / '__init__.py').exists(), f'repo not found: {REPO}'
if str(REPO) not in _sys.path:
    _sys.path.insert(0, str(REPO))
import superfermion._sf_core as _sfcore
_core = _Path(_sfcore.__file__)
print('=' * 78)
print('ENGINE PROVENANCE - fixed local build pinned for this run')
print('=' * 78)
print(f'  engine : {_core.name}  md5 {_hashlib.md5(_core.read_bytes()).hexdigest().upper()}')
print()

import gc
import time
import json
import platform
import numpy as np
from statistics import median

import superfermion as sf
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

print(f"Python: {platform.python_version()}")
print(f"CPU: {platform.processor() or 'unknown'}")
print(f"OS: {platform.system()} {platform.release()}")
print(f"SF version: {sf.__version__}")
try:
    import qiskit; print(f"Qiskit: {qiskit.__version__}")
except: pass
try:
    import qiskit_aer; print(f"Qiskit Aer: {qiskit_aer.__version__}")
except: pass

ENGINE PROVENANCE - fixed local build pinned for this run
  engine : _sf_core.cp313-win_amd64.pyd  md5 F8081676682427D2C14853AA79923388



Python: 3.13.3
CPU: Intel64 Family 6 Model 142 Stepping 12, GenuineIntel
OS: Windows 11
SF version: 0.1.12
Qiskit: 2.1.2
Qiskit Aer: 0.17.2


In [2]:
N_TRIALS = 15

def timed(fn):
    gc.disable()
    t0 = time.perf_counter()
    result = fn()
    elapsed = (time.perf_counter() - t0) * 1000
    gc.enable()
    return result, elapsed

def benchmark(fn, n_trials=N_TRIALS):
    times = []
    for _ in range(n_trials):
        _, t = timed(fn)
        times.append(round(t, 2))
    med = median(times)
    s = sorted(times)
    q1 = s[len(s)//4]
    q3 = s[3*len(s)//4]
    return med, times, q1, q3

def fidelity(sv1, sv2):
    return float(abs(np.vdot(sv1, sv2)) ** 2)

def fmt(sf_ms, qk_ms):
    r = qk_ms / sf_ms if sf_ms > 0 else float('inf')
    return f'SF {r:.1f}x faster' if r > 1 else f'Qiskit {1/r:.1f}x faster'

print('Helpers defined.')

Helpers defined.


In [3]:
def build_hardware_efficient(n, depth=3, seed=42):
    rng = np.random.RandomState(seed)
    angles = rng.uniform(0, 2*np.pi, size=(depth, n, 2))
    sc = sf.Circuit(n)
    qc = QuantumCircuit(n)
    for d in range(depth):
        for q in range(n):
            sc.ry(angles[d, q, 0], q); qc.ry(angles[d, q, 0], q)
            sc.rz(angles[d, q, 1], q); qc.rz(angles[d, q, 1], q)
        for q in range(0, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
        for q in range(1, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
    return sc, qc

def build_ghz(n):
    sc = sf.Circuit(n); qc = QuantumCircuit(n)
    sc.h(0); qc.h(0)
    for i in range(1, n): sc.cx(0, i); qc.cx(0, i)
    return sc, qc

def build_clifford_only(n):
    sc = sf.Circuit(n); qc = QuantumCircuit(n)
    for q in range(n):         sc.h(q);  qc.h(q)
    for q in range(0, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
    for q in range(n):         sc.s(q);  qc.s(q)
    for q in range(1, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
    for q in range(n):         sc.h(q);  qc.h(q)
    for q in range(0, n-1, 2): sc.cx(q, q+1); qc.cx(q, q+1)
    return sc, qc

print('Circuit builders defined.')

Circuit builders defined.


## 1. Statevector Benchmark

In [4]:
sv_qubit_counts = [10, 12, 14, 16, 18, 20, 22]
sv_results = {}
qk_sim = AerSimulator(method='statevector')

print('=' * 75)
print('STATEVECTOR BENCHMARK (Hardware-Efficient Ansatz, depth=3)')
print('=' * 75)

# Warmup both
n_max = sv_qubit_counts[-1]
print(f'Warming up at n={n_max}...')
_ = sf.run(build_hardware_efficient(n_max)[0], device='cpu', shots=0)
qc_w = build_hardware_efficient(n_max)[1]; qc_w.save_statevector()
_ = qk_sim.run(qc_w).result()
_ = sf.run(build_hardware_efficient(n_max)[0], device='cpu', shots=0)
_ = qk_sim.run(qc_w).result()
print('Done.\n')

for n in sv_qubit_counts:
    # Correctness
    sc_c, qc_c = build_hardware_efficient(n)
    sf_sv = np.array(sf.run(sc_c, device='cpu', shots=0).statevector)
    qc_c.save_statevector()
    qk_sv = np.array(qk_sim.run(qc_c).result().get_statevector(), dtype=np.complex128)
    fid = fidelity(sf_sv, qk_sv)
    assert fid > 0.9999, f'Fidelity fail n={n}: {fid}'

    # Per-size warmup
    _ = sf.run(build_hardware_efficient(n)[0], device='cpu', shots=0)
    qc_wu = build_hardware_efficient(n)[1]; qc_wu.save_statevector()
    _ = qk_sim.run(qc_wu).result()

    # SF
    def run_sf(n=n):
        sc, _ = build_hardware_efficient(n)
        return sf.run(sc, device='cpu', shots=0)
    sf_med, sf_times, sf_q1, sf_q3 = benchmark(run_sf)

    # Qiskit
    def run_qk(n=n):
        _, qc = build_hardware_efficient(n)
        qc.save_statevector()
        return qk_sim.run(qc).result()
    qk_med, qk_times, qk_q1, qk_q3 = benchmark(run_qk)

    sv_results[n] = {
        'sf_ms': sf_med, 'qk_ms': qk_med,
        'sf_all': sf_times, 'qk_all': qk_times,
        'sf_iqr': [sf_q1, sf_q3], 'qk_iqr': [qk_q1, qk_q3],
        'fidelity': fid,
    }
    print(f'n={n:2d} | SF {sf_med:8.1f}ms [{sf_q1:.1f}–{sf_q3:.1f}] | '
          f'Qiskit {qk_med:8.1f}ms [{qk_q1:.1f}–{qk_q3:.1f}] | '
          f'fid={fid:.6f} | {fmt(sf_med, qk_med)}')

STATEVECTOR BENCHMARK (Hardware-Efficient Ansatz, depth=3)
Warming up at n=22...


Done.



n=10 | SF      5.5ms [5.4–5.8] | Qiskit      8.3ms [6.9–9.6] | fid=1.000000 | SF 1.5x faster


n=12 | SF      8.3ms [6.2–9.3] | Qiskit      9.8ms [8.8–10.4] | fid=1.000000 | SF 1.2x faster


n=14 | SF      3.6ms [3.2–4.5] | Qiskit     15.8ms [15.2–17.3] | fid=1.000000 | SF 4.4x faster


n=16 | SF      7.7ms [6.8–8.0] | Qiskit     26.5ms [24.6–29.9] | fid=1.000000 | SF 3.5x faster


n=18 | SF     23.7ms [20.6–24.8] | Qiskit     67.2ms [64.8–71.3] | fid=1.000000 | SF 2.8x faster


n=20 | SF     76.7ms [75.2–78.8] | Qiskit    274.6ms [262.6–281.7] | fid=1.000000 | SF 3.6x faster


n=22 | SF    299.3ms [297.6–309.9] | Qiskit   1013.7ms [1008.5–1026.3] | fid=1.000000 | SF 3.4x faster


## 2. Stabilizer Benchmark

Correctness: compare SF stabilizer samples against the **exact** statevector distribution (not another sample). TVD threshold is 4-sigma of the expected sampling noise for a multinomial.

In [5]:
stab_qubit_counts = [10, 20, 50, 100, 200, 500]
stab_results = {}
qk_stab = AerSimulator(method='stabilizer')

print('=' * 75)
print('STABILIZER BENCHMARK (Clifford circuit, shots=10000)')
print('Correctness: TVD of SF stabilizer vs exact SV distribution')
print('=' * 75)

# Warmup
sc_w, qc_w = build_clifford_only(stab_qubit_counts[-1])
print(f'Warming up at n={stab_qubit_counts[-1]}...')
try:
    _ = sf.run(sc_w, device='cpu', method='stabilizer', shots=1000)
    sf_stab_ok = True
except Exception as e:
    print(f'SF stabilizer unavailable: {e}')
    sf_stab_ok = False
qc_w_m = qc_w.copy(); qc_w_m.measure_all()
_ = qk_stab.run(qc_w_m, shots=1000).result()
print('Done.\n')

for n in stab_qubit_counts:
    corr = ''
    if sf_stab_ok and n <= 16:
        exact_sv = np.array(sf.run(build_clifford_only(n)[0], device='cpu', shots=0).statevector)
        exact_probs = {format(i, f'0{n}b'): float(p) for i, p in enumerate(np.abs(exact_sv)**2) if p > 1e-15}
        n_check = 200000
        sf_c = sf.run(build_clifford_only(n)[0], device='cpu', method='stabilizer', shots=n_check).counts
        all_keys = set(sf_c.keys()) | set(exact_probs.keys())
        dist = 0.5 * sum(abs(sf_c.get(k,0)/n_check - exact_probs.get(k,0)) for k in all_keys)
        k = len(exact_probs)
        expected = np.sqrt(k / (2 * np.pi * n_check))
        threshold = expected * 4
        corr = f'TVD={dist:.4f} (expect~{expected:.4f}, thr={threshold:.4f})'
        assert dist < threshold, f'Stabilizer mismatch n={n}: {corr}'

    # Per-size warmup
    if sf_stab_ok:
        _ = sf.run(build_clifford_only(n)[0], device='cpu', method='stabilizer', shots=1000)
    qc_wu = build_clifford_only(n)[1]; qc_wu.measure_all()
    _ = qk_stab.run(qc_wu, shots=1000).result()

    if sf_stab_ok:
        def run_sf_s(n=n):
            return sf.run(build_clifford_only(n)[0], device='cpu', method='stabilizer', shots=10000)
        sf_med, sf_times, sf_q1, sf_q3 = benchmark(run_sf_s)
    else:
        sf_med, sf_times, sf_q1, sf_q3 = float('nan'), [], float('nan'), float('nan')

    def run_qk_s(n=n):
        _, qc = build_clifford_only(n)
        qc.measure_all()
        return qk_stab.run(qc, shots=10000).result()
    qk_med, qk_times, qk_q1, qk_q3 = benchmark(run_qk_s)

    stab_results[n] = {
        'sf_ms': sf_med, 'qk_ms': qk_med,
        'sf_all': sf_times, 'qk_all': qk_times,
        'sf_iqr': [sf_q1, sf_q3], 'qk_iqr': [qk_q1, qk_q3],
        'correctness': corr,
    }
    if sf_stab_ok and not np.isnan(sf_med):
        print(f'n={n:3d} | SF {sf_med:8.1f}ms [{sf_q1:.1f}–{sf_q3:.1f}] | '
              f'Qiskit {qk_med:8.1f}ms [{qk_q1:.1f}–{qk_q3:.1f}] | '
              f'{fmt(sf_med, qk_med)} | {corr}')
    else:
        print(f'n={n:3d} | SF N/A | Qiskit {qk_med:8.1f}ms [{qk_q1:.1f}–{qk_q3:.1f}]')

STABILIZER BENCHMARK (Clifford circuit, shots=10000)
Correctness: TVD of SF stabilizer vs exact SV distribution
Warming up at n=500...


Done.



n= 10 | SF      4.7ms [4.4–5.1] | Qiskit    129.7ms [127.4–133.4] | SF 27.5x faster | TVD=0.0281 (expect~0.0285, thr=0.1142)


n= 20 | SF     16.7ms [13.1–18.1] | Qiskit    231.7ms [227.0–240.5] | SF 13.8x faster | 


n= 50 | SF     20.3ms [18.6–25.1] | Qiskit    784.2ms [752.1–889.1] | SF 38.7x faster | 


n=100 | SF     23.0ms [22.0–28.9] | Qiskit   1991.0ms [1960.2–2031.5] | SF 86.7x faster | 


n=200 | SF     33.7ms [32.6–37.1] | Qiskit   7730.4ms [7496.8–9012.0] | SF 229.1x faster | 


n=500 | SF     90.7ms [89.8–100.8] | Qiskit  52273.9ms [51481.5–56593.8] | SF 576.3x faster | 


## 3. MPS Benchmark

GHZ circuits (low entanglement). Correctness: must produce ~50/50 on |0...0> and |1...1>.

In [6]:
mps_qubit_counts = [10, 20, 30, 50, 80, 100]
mps_results = {}
qk_mps = AerSimulator(method='matrix_product_state')

print('=' * 75)
print('MPS BENCHMARK (GHZ circuit, shots=10000, bond_dim=64)')
print('Correctness: GHZ distribution must be ~50/50 on |0..0> and |1..1>')
print('=' * 75)

# Warmup
sc_w, qc_w = build_ghz(mps_qubit_counts[-1])
print(f'Warming up at n={mps_qubit_counts[-1]}...')
try:
    _ = sf.run(sc_w, device='cpu', method='mps', shots=1000, bond_dim=64)
    sf_mps_ok = True
except Exception as e:
    print(f'SF MPS unavailable: {e}')
    sf_mps_ok = False
qc_w_m = qc_w.copy(); qc_w_m.measure_all()
_ = qk_mps.run(qc_w_m, shots=1000).result()
print('Done.\n')

for n in mps_qubit_counts:
    corr = ''
    if sf_mps_ok and n <= 50:
        sc_c, _ = build_ghz(n)
        n_check = 100000
        sf_c = sf.run(sc_c, device='cpu', method='mps', shots=n_check, bond_dim=64).counts
        z = '0' * n; o = '1' * n
        total = sum(sf_c.values())
        fz = sf_c.get(z, 0) / total if total > 0 else 0
        fo = sf_c.get(o, 0) / total if total > 0 else 0
        err = abs(fz - 0.5) + abs(fo - 0.5)
        corr = f'GHZ err={err:.4f} (|0>={fz:.3f}, |1>={fo:.3f})'
        assert err < 0.03, f'GHZ mismatch n={n}: {corr}'

    # Per-size warmup
    if sf_mps_ok:
        _ = sf.run(build_ghz(n)[0], device='cpu', method='mps', shots=1000, bond_dim=64)
    qc_wu = build_ghz(n)[1]; qc_wu.measure_all()
    _ = qk_mps.run(qc_wu, shots=1000).result()

    if sf_mps_ok:
        def run_sf_m(n=n):
            return sf.run(build_ghz(n)[0], device='cpu', method='mps', shots=10000, bond_dim=64)
        sf_med, sf_times, _, _ = benchmark(run_sf_m)
    else:
        sf_med, sf_times = float('nan'), []

    def run_qk_m(n=n):
        _, qc = build_ghz(n)
        qc.measure_all()
        return qk_mps.run(qc, shots=10000).result()
    qk_med, qk_times, _, _ = benchmark(run_qk_m)

    mps_results[n] = {'sf_ms': sf_med, 'qk_ms': qk_med, 'sf_all': sf_times, 'qk_all': qk_times, 'correctness': corr}
    if sf_mps_ok and not np.isnan(sf_med):
        print(f'n={n:3d} | SF {sf_med:8.1f}ms | Qiskit {qk_med:8.1f}ms | {fmt(sf_med, qk_med)} | {corr}')
    else:
        print(f'n={n:3d} | SF N/A | Qiskit {qk_med:8.1f}ms')

MPS BENCHMARK (GHZ circuit, shots=10000, bond_dim=64)
Correctness: GHZ distribution must be ~50/50 on |0..0> and |1..1>
Warming up at n=100...


Done.



n= 10 | SF     20.7ms | Qiskit    325.4ms | SF 15.7x faster | GHZ err=0.0022 (|0>=0.501, |1>=0.499)


n= 20 | SF     32.6ms | Qiskit    631.4ms | SF 19.4x faster | GHZ err=0.0022 (|0>=0.501, |1>=0.499)


n= 30 | SF     50.1ms | Qiskit    952.6ms | SF 19.0x faster | GHZ err=0.0022 (|0>=0.501, |1>=0.499)


n= 50 | SF     76.0ms | Qiskit   1658.7ms | SF 21.8x faster | GHZ err=0.0022 (|0>=0.501, |1>=0.499)


n= 80 | SF    136.8ms | Qiskit   2817.3ms | SF 20.6x faster | 


n=100 | SF    232.0ms | Qiskit   3543.9ms | SF 15.3x faster | 


## 4. Shot-Based Sampling

In [7]:
shot_qubit_counts = [10, 14, 18, 20, 22]
shot_counts_list = [1000, 10000, 100000]
shot_results = {}
qk_sim = AerSimulator(method='statevector')

print('=' * 75)
print('SHOT-BASED SAMPLING (HE depth=3, Rust-native simulate_and_sample)')
print('Correctness: statevector fidelity check at each qubit count')
print('=' * 75)

for n in shot_qubit_counts:
    sc_c, qc_c = build_hardware_efficient(n)
    sf_sv = np.array(sf.run(sc_c, device='cpu', shots=0).statevector)
    qc_c.save_statevector()
    qk_sv = np.array(qk_sim.run(qc_c).result().get_statevector(), dtype=np.complex128)
    fid = fidelity(sf_sv, qk_sv)
    assert fid > 0.9999, f'Shot fidelity fail n={n}: {fid}'
print(f'Fidelity OK at all sizes (same circuit as statevector section).\n')

for n_shots in shot_counts_list:
    print(f'--- shots={n_shots} ---')
    for n in shot_qubit_counts:
        # Per-size warmup
        _ = sf.run(build_hardware_efficient(n)[0], device='cpu', shots=min(n_shots, 1000), return_statevector=False)
        qc_wu = build_hardware_efficient(n)[1]; qc_wu.measure_all()
        _ = qk_sim.run(qc_wu, shots=min(n_shots, 1000)).result()

        def run_sf_sh(n=n, ns=n_shots):
            return sf.run(build_hardware_efficient(n)[0], device='cpu', shots=ns, return_statevector=False)
        sf_med, sf_times, _, _ = benchmark(run_sf_sh)

        def run_qk_sh(n=n, ns=n_shots):
            _, qc = build_hardware_efficient(n)
            qc.measure_all()
            return qk_sim.run(qc, shots=ns).result()
        qk_med, qk_times, _, _ = benchmark(run_qk_sh)

        shot_results[(n, n_shots)] = {'sf_ms': sf_med, 'qk_ms': qk_med, 'sf_all': sf_times, 'qk_all': qk_times}
        print(f'  n={n:2d} | SF {sf_med:8.1f}ms | Qiskit {qk_med:8.1f}ms | {fmt(sf_med, qk_med)}')

SHOT-BASED SAMPLING (HE depth=3, Rust-native simulate_and_sample)
Correctness: statevector fidelity check at each qubit count


Fidelity OK at all sizes (same circuit as statevector section).

--- shots=1000 ---


  n=10 | SF      4.5ms | Qiskit     14.7ms | SF 3.2x faster


  n=14 | SF      7.2ms | Qiskit     30.6ms | SF 4.3x faster


  n=18 | SF     25.2ms | Qiskit     75.5ms | SF 3.0x faster


...

  n=10 | SF     32.8ms | Qiskit    290.1ms | SF 8.8x faster


  n=14 | SF     34.8ms | Qiskit    316.9ms | SF 9.1x faster


  n=18 | SF     93.5ms | Qiskit    387.4ms | SF 4.1x faster


  n=20 | SF    178.4ms | Qiskit    635.7ms | SF 3.6x faster


  n=22 | SF    439.5ms | Qiskit   1571.0ms | SF 3.6x faster


## 5. Depth Scaling

In [8]:
depth_n = 16
depths = [1, 3, 5, 10, 20, 50]
depth_results = {}
qk_sim = AerSimulator(method='statevector')

print('=' * 75)
print(f'DEPTH SCALING (n={depth_n}, Hardware-Efficient)')
print('Correctness: fidelity check at each depth')
print('=' * 75)

for d in depths:
    sc_c, qc_c = build_hardware_efficient(depth_n, depth=d)
    sf_sv = np.array(sf.run(sc_c, device='cpu', shots=0).statevector)
    qc_c.save_statevector()
    qk_sv = np.array(qk_sim.run(qc_c).result().get_statevector(), dtype=np.complex128)
    fid = fidelity(sf_sv, qk_sv)
    assert fid > 0.9999, f'Depth fidelity fail d={d}: {fid}'

    # Per-depth warmup
    _ = sf.run(build_hardware_efficient(depth_n, depth=d)[0], device='cpu', shots=0)
    qc_wu = build_hardware_efficient(depth_n, depth=d)[1]; qc_wu.save_statevector()
    _ = qk_sim.run(qc_wu).result()

    def run_sf_d(d=d):
        return sf.run(build_hardware_efficient(depth_n, depth=d)[0], device='cpu', shots=0)
    sf_med, sf_times, sf_q1, sf_q3 = benchmark(run_sf_d)

    def run_qk_d(d=d):
        _, qc = build_hardware_efficient(depth_n, depth=d)
        qc.save_statevector()
        return qk_sim.run(qc).result()
    qk_med, qk_times, qk_q1, qk_q3 = benchmark(run_qk_d)

    ng = build_hardware_efficient(depth_n, depth=d)[0].gate_count
    depth_results[d] = {
        'sf_ms': sf_med, 'qk_ms': qk_med,
        'sf_all': sf_times, 'qk_all': qk_times,
        'sf_iqr': [sf_q1, sf_q3], 'qk_iqr': [qk_q1, qk_q3],
        'n_gates': ng, 'fidelity': fid,
    }
    print(f'depth={d:2d} ({ng:4d} gates) | SF {sf_med:8.1f}ms [{sf_q1:.1f}–{sf_q3:.1f}] | '
          f'Qiskit {qk_med:8.1f}ms [{qk_q1:.1f}–{qk_q3:.1f}] | '
          f'fid={fid:.6f} | {fmt(sf_med, qk_med)}')

DEPTH SCALING (n=16, Hardware-Efficient)
Correctness: fidelity check at each depth


depth= 1 (  47 gates) | SF      4.2ms [4.0–4.4] | Qiskit     11.7ms [9.8–12.6] | fid=1.000000 | SF 2.8x faster


depth= 3 ( 141 gates) | SF      9.1ms [7.2–9.3] | Qiskit     30.6ms [29.7–33.4] | fid=1.000000 | SF 3.4x faster


depth= 5 ( 235 gates) | SF     12.7ms [10.3–14.1] | Qiskit     44.6ms [43.4–46.4] | fid=1.000000 | SF 3.5x faster


depth=10 ( 470 gates) | SF     18.1ms [17.1–23.2] | Qiskit     90.9ms [88.5–105.5] | fid=1.000000 | SF 5.0x faster


depth=20 ( 940 gates) | SF     32.2ms [31.3–33.0] | Qiskit    222.4ms [175.6–235.3] | fid=1.000000 | SF 6.9x faster


depth=50 (2350 gates) | SF     83.7ms [79.0–86.5] | Qiskit    361.9ms [353.8–370.7] | fid=1.000000 | SF 4.3x faster


## Summary Plot

In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Superfermion vs Qiskit Aer — CPU (cache-cleared, honest)', fontsize=14, fontweight='bold')

ax = axes[0, 0]
ns = sorted(sv_results.keys())
ax.semilogy(ns, [sv_results[n]['sf_ms'] for n in ns], 'o-', label='SF', color='#2196F3', lw=2)
ax.semilogy(ns, [sv_results[n]['qk_ms'] for n in ns], 's-', label='Qiskit', color='#FF5722', lw=2)
ax.set_xlabel('Qubits'); ax.set_ylabel('ms (log)'); ax.set_title('Statevector (HE, depth=3)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ns2 = sorted(stab_results.keys())
if sf_stab_ok:
    ax.semilogy(ns2, [stab_results[n]['sf_ms'] for n in ns2], 'o-', label='SF', color='#2196F3', lw=2)
ax.semilogy(ns2, [stab_results[n]['qk_ms'] for n in ns2], 's-', label='Qiskit', color='#FF5722', lw=2)
ax.set_xlabel('Qubits'); ax.set_ylabel('ms (log)'); ax.set_title('Stabilizer (Clifford, 10k shots)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ns3 = sorted(mps_results.keys())
if sf_mps_ok:
    ax.semilogy(ns3, [mps_results[n]['sf_ms'] for n in ns3], 'o-', label='SF', color='#2196F3', lw=2)
ax.semilogy(ns3, [mps_results[n]['qk_ms'] for n in ns3], 's-', label='Qiskit', color='#FF5722', lw=2)
ax.set_xlabel('Qubits'); ax.set_ylabel('ms (log)'); ax.set_title('MPS (GHZ, 10k shots)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ds = sorted(depth_results.keys())
ax.semilogy(ds, [depth_results[d]['sf_ms'] for d in ds], 'o-', label='SF', color='#2196F3', lw=2)
ax.semilogy(ds, [depth_results[d]['qk_ms'] for d in ds], 's-', label='Qiskit', color='#FF5722', lw=2)
ax.set_xlabel('Depth'); ax.set_ylabel('ms (log)'); ax.set_title(f'Depth Scaling (n={depth_n})')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('benchmark_master_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved benchmark_master_results.png')

Saved benchmark_master_results.png


C:\Users\ASUS\AppData\Local\Temp\ipykernel_7752\165511098.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
all_data = {
    'statevector': {str(k): v for k, v in sv_results.items()},
    'stabilizer': {str(k): v for k, v in stab_results.items()},
    'mps': {str(k): v for k, v in mps_results.items()},
    'shots': {str(k): v for k, v in shot_results.items()},
    'depth': {str(k): v for k, v in depth_results.items()},
}
with open('benchmark_master_data.json', 'w') as f:
    json.dump(all_data, f, indent=2, default=str)
print('Saved benchmark_master_data.json')

Saved benchmark_master_data.json


## Methodology Notes

### No caching
SF does not cache simulation results. Each timed call performs a full simulation from scratch.

### Correctness verification
- **Statevector**: Cross-framework fidelity (SF vs Qiskit Aer, direct comparison — both use LSB qubit ordering). Fidelity = 1.0 required.
- **Stabilizer**: TVD of SF stabilizer samples against exact statevector distribution. Threshold is 4-sigma of expected multinomial sampling noise: `4 * sqrt(K / (2*pi*N))` where K = number of outcomes, N = shots.
- **MPS**: GHZ state check — deviation from 50/50 split on |0...0> and |1...1> must be < 0.03.

### Known limitations
- WSL2 introduces scheduling noise; Qiskit times may have non-monotonic spikes
- MPS benchmark uses GHZ (low entanglement) — highly entangled circuits would show different scaling
- n=24 excluded from statevector to keep runtime under 15 minutes